In [ ]:
# the idea of this notebook is to get an understanding on the number of digital native vs analogue original of protected townscapes of the Dutch Heritage Agency (RCE) 

import re
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
# Panda settings for showing data (this is foremost done to more easily explore the data while processing it)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# the following has been created with the help of chatgpt
# --------------------------
# 1) Load Townscapes WFS -> DataFrame
# --------------------------

WFS_URL = (
    "https://services.rce.geovoorziening.nl/rce/wfs"
    "?request=GetFeature&service=WFS&version=1.1.0"
    "&outputFormat=application%2Fjson&typeName=Townscapes"
)

def load_townscapes_df(wfs_url: str = WFS_URL) -> pd.DataFrame:
    r = requests.get(wfs_url, timeout=60)
    r.raise_for_status()
    data = r.json()

    # properties live under features[].properties
    df = pd.json_normalize(data["features"], sep="_")

    # Commonly useful:
    # - properties_* columns are the attributes
    # - geometry_* columns contain geometry metadata
    # If you only want attributes, uncomment the next line:
    # df = pd.json_normalize([f["properties"] for f in data["features"]])

    # Make sure the column is exactly named KICHLINK
    # (in this dataset it already is, as properties_KICHLINK if you normalized whole features)
    if "KICHLINK" not in df.columns and "properties_KICHLINK" in df.columns:
        df = df.rename(columns={"properties_KICHLINK": "KICHLINK"})

    return df


# --------------------------
# 2) URL checking logic (booleans + urls + list)
# --------------------------

include_classic_pdf = True

VARIANTS = [
    ("begrenzingskaart_pdf",  "https://cultureelerfgoed.info/Beschermde_Gezichten/BG{n}/BEGRENZINGSKAART_aanwijzing_{id}.pdf"),
    ("begrenzingskaart_jpg",  "https://cultureelerfgoed.info/Beschermde_Gezichten/BG{n}/BEGRENZINGSKAART_aanwijzing_{id}.jpg"),
    ("digitaal_pdf",          "https://cultureelerfgoed.info/Beschermde_Gezichten/BG{n}/DIGITAAL_BEGRENZINGSKAART_aanwijzing_{id}.pdf"),
    ("digitaal_jpg",          "https://cultureelerfgoed.info/Beschermde_Gezichten/BG{n}/DIGITAAL_BEGRENZINGSKAART_aanwijzing_{id}.jpg"),
]
if not include_classic_pdf:
    VARIANTS = [v for v in VARIANTS if v[0] != "begrenzingskaart_pdf"]


def extract_gezicht_id(kich_link: str) -> str | None:
    if not isinstance(kich_link, str):
        return None
    m = re.search(r"/Gezicht/(\d+)\s*$", kich_link)
    return m.group(1) if m else None


def bg_folder(numeric_id: str) -> str:
    # Adjust here if the site uses zero-padding (e.g., BG0001)
    return numeric_id


def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("HEAD", "GET"),
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    s.headers.update({"User-Agent": "KICHLINK-check/1.0"})
    return s


def url_exists(session: requests.Session, url: str, timeout: float = 20.0) -> bool:
    try:
        r = session.head(url, allow_redirects=True, timeout=timeout)
        if r.status_code == 200:
            return True
        if r.status_code in (403, 405):
            r = session.get(url, stream=True, allow_redirects=True, timeout=timeout)
            return r.status_code == 200
        return False
    except requests.RequestException:
        return False


def check_kich_links(df: pd.DataFrame, link_col: str = "KICHLINK") -> tuple[pd.DataFrame, pd.DataFrame]:
    session = make_session()

    df_out = df.copy()
    df_out["gezicht_id"] = df_out[link_col].apply(extract_gezicht_id)

    # Add boolean + url columns per variant
    for key, _ in VARIANTS:
        df_out[f"has_{key}"] = False
        df_out[f"url_{key}"] = pd.NA

    found_urls_col: list[list[str]] = []
    records_long = []

    for idx, gid in df_out["gezicht_id"].items():
        if not gid:
            found_urls_col.append([])
            continue

        n = bg_folder(gid)
        found = []

        for key, tpl in VARIANTS:
            url = tpl.format(n=n, id=gid)
            exists = url_exists(session, url)

            df_out.at[idx, f"has_{key}"] = bool(exists)
            if exists:
                df_out.at[idx, f"url_{key}"] = url
                found.append(url)
                records_long.append({
                    "row_index": idx,
                    link_col: df_out.at[idx, link_col],
                    "gezicht_id": gid,
                    "variant": key,
                    "asset_url": url,
                    "asset_type": url.rsplit(".", 1)[-1].lower(),
                })

        found_urls_col.append(found)

    df_out["found_urls"] = found_urls_col
    found_long = pd.DataFrame.from_records(records_long)
    return df_out, found_long


# --------------------------
# 3) Run it
# --------------------------
if __name__ == "__main__":
    df = load_townscapes_df()
    df_checked, found_assets = check_kich_links(df, link_col="KICHLINK")

    # Save results if you like:
    # df_checked.to_parquet("townscapes_checked.parquet", index=False)
    # found_assets.to_csv("townscapes_assets_found.csv", index=False)

    print(df_checked.filter(
        ["KICHLINK", "gezicht_id", "found_urls"] +
        [c for c in df_checked.columns if c.startswith("has_")] +
        [c for c in df_checked.columns if c.startswith("url_")]
    ).head(3))

In [ ]:
scanned_pdf = df_checked["has_begrenzingskaart_pdf"].sum()

In [ ]:
scanned_jpg = df_checked["has_begrenzingskaart_jpg"].sum()

In [ ]:
digi_native_jpg = df_checked["has_digitaal_jpg"].sum()

In [ ]:
digi_native_pdf = df_checked["has_digitaal_pdf"].sum()

In [ ]:
scanned = scanned_jpg + scanned_pdf
print(scanned)

In [ ]:
digi_native = digi_native_pdf + digi_native_jpg
print(digi_native)

In [ ]:
df_missing = df_checked[df_checked["found_urls"].apply(lambda x: isinstance(x, list) and len(x) == 0)].copy()

In [ ]:
df_missing.shape

In [ ]:
n_duplicates = df_checked["gezicht_id"].duplicated().sum()

In [ ]:
n_duplicates

In [ ]:
scanned+digi_native+54
# gezicht_id = 1474 has both a scanned .jpg and .pdf